# 📈 Phase 1: Feature Analysis & Selection
**Project:** Machine Learning Experimental Design — Random Forest Churn Prediction

This notebook focuses on analyzing the relationships between different independent features, identifying multicollinearity, checking for data leakage, and applying deterministic feature selection.

In [ ]:
import sys
from pathlib import Path

# Ensure project modules are importable
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_data
from src.feature_analysis import (
    compute_correlation_matrix,
    plot_correlation_heatmap,
    remove_highly_correlated_features,
    check_data_leakage
)

sns.set_theme(style="whitegrid")

## 1. Load Preprocessed Data
We load features `X` and target `y` using our custom pipeline, which automatically drops high-cardinality nominal identifiers (`state`, `area_code`) and encodes binary columns.

In [ ]:
X, y = load_data()
print(f"X shape: {X.shape}")
X.head()

## 2. Check for Potential Data Leakage
We perform a heuristic scan to identify if any variable has an extremely high correlation ($r \geq 0.95$) with the target variable `churn` which could signal data leakage (e.g. features recorded after the event occurred).

In [ ]:
df_full = X.copy()
df_full['churn'] = y.values
leaky_cols = check_data_leakage(df_full, 'churn', threshold=0.95)
print(f"Leaky features detected: {leaky_cols}")

## 3. Visualize Feature Correlations (Multicollinearity Heatmap)

In [ ]:
plot_correlation_heatmap(X, figsize=(14, 12))

## 4. Remove Highly Correlated Features
To prevent multicollinearity, we identify feature pairs with correlation above 0.95 and drop the column with higher mean absolute correlation with all other features.

In [ ]:
X_reduced, dropped_cols = remove_highly_correlated_features(X, threshold=0.95)
print(f"Dropped {len(dropped_cols)} features: {dropped_cols}")
print(f"Reduced feature matrix shape: {X_reduced.shape}")

## 📌 Key Insights on Feature Redundancy

The deterministic feature selection automatically drops:
1. **`total_day_charge`**: Correlates perfectly ($1.00$) with `total_day_minutes`.
2. **`total_eve_charge`**: Correlates perfectly ($1.00$) with `total_eve_minutes`.
3. **`total_night_minutes`**: Correlates perfectly ($1.00$) with `total_night_charge`.
4. **`total_intl_charge`**: Correlates perfectly ($1.00$) with `total_intl_minutes`.
5. **`voice_mail_plan`**: Highly correlated with `number_vmail_messages` (if a plan is inactive, messages count is always 0).

By dropping these redundant charges/plans, we successfully resolve **multicollinearity**, reduce feature space complexity, and make model interpretation statistically sound.